# Assemblée Nationale — IDEAL Point Model (17e législature)

Miroir du notebook Sénat `senat_ideal_point_model_R_simple.ipynb`.

Modèle unique : `pscl::ideal` (2 dimensions) sur les scrutins publics nominatifs de la 17e législature.
Données préparées dans `data/assemblee/model_ready/` à partir des dumps officiels data.assemblee-nationale.fr.


## 1. Chargement


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)
options(digits = 3)

library(tidyverse)
library(pscl)
theme_set(theme_bw())

# Chemins relatifs au dépôt (éviter les chemins absolus avec espaces)
data_folder <- file.path("..", "data", "assemblee", "model_ready")
out_dir <- file.path("..", "data", "assemblee", "outputs")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

votes <- read_csv(file.path(data_folder, "votes_deputes_actifs_l17.csv.gz"), show_col_types = FALSE)
deputes <- read_csv(file.path(data_folder, "deputes_actifs_l17.csv"), show_col_types = FALSE)
rollcalls <- read_csv(file.path(data_folder, "scrutins_rollcalls_l17.csv"), show_col_types = FALSE)

dim(votes)
dim(deputes)
dim(rollcalls)


## 2. Matrice de votes

Filtres identiques au Sénat : pour/contre uniquement, minorité ≥ 10 %, ≥ 25 votes par député.


In [ ]:
votes_active <- votes %>% semi_join(deputes, by = "acteur_id")

votes_binary <- votes_active %>%
  filter(vote_value %in% c(1, -1)) %>%
  mutate(vote_model = if_else(vote_value == 1, 1L, 0L))

vote_balance <- votes_binary %>%
  group_by(scrutin_id) %>%
  summarise(
    yes_votes = sum(vote_model == 1L),
    no_votes = sum(vote_model == 0L),
    total_votes = n(),
    minority_share = pmin(yes_votes, no_votes) / total_votes,
    .groups = "drop"
  )

good_votes <- vote_balance %>% filter(minority_share >= 0.10)
votes_filtered <- votes_binary %>% semi_join(good_votes, by = "scrutin_id")
good_deputes <- votes_filtered %>% count(acteur_id, name = "n_votes") %>% filter(n_votes >= 25)
votes_model <- votes_filtered %>% semi_join(good_deputes, by = "acteur_id")

tibble(
  active_deputes = n_distinct(votes_active$acteur_id),
  model_deputes = n_distinct(votes_model$acteur_id),
  model_roll_calls = n_distinct(votes_model$scrutin_id),
  model_rows = nrow(votes_model)
)


In [ ]:
vote_matrix_data <- votes_model %>%
  select(acteur_id, scrutin_id, vote_model) %>%
  group_by(acteur_id, scrutin_id) %>%
  summarise(vote_model = first(vote_model), .groups = "drop") %>%
  pivot_wider(names_from = scrutin_id, values_from = vote_model) %>%
  arrange(acteur_id)

deputy_ids <- as.character(vote_matrix_data$acteur_id)
vote_matrix <- vote_matrix_data %>% select(-acteur_id) %>% as.matrix()
rownames(vote_matrix) <- deputy_ids
colnames(vote_matrix) <- make.names(colnames(vote_matrix), unique = TRUE)

deputy_info <- tibble(acteur_id = deputy_ids) %>%
  left_join(
    deputes %>% mutate(acteur_id = as.character(acteur_id)) %>%
      select(acteur_id, nom, prenom, groupe_code, groupe_libelle_court),
    by = "acteur_id"
  ) %>%
  mutate(full_name = paste(prenom, nom))

an_rollcall <- rollcall(
  vote_matrix, yea = 1, nay = 0, missing = NA,
  legis.names = deputy_ids, vote.names = colnames(vote_matrix),
  legis.data = deputy_info
)
summary(an_rollcall)


## 3. Estimation `pscl::ideal`

Paramètres alignés sur le notebook Sénat : `d=2`, `maxiter=1000`, `burnin=500`, `thin=25`, `impute=FALSE`, `seed=123`.


In [ ]:
left_groups <- c("LFI-NFP", "GDR", "SOC", "ECOS")
right_groups <- c("RN", "DR", "UDDPLR", "UDR")
group_order <- c("LFI-NFP", "GDR", "ECOS", "SOC", "LIOT", "NI", "EPR", "DEM", "HOR", "DR", "UDDPLR", "UDR", "RN")

# Senate convention: dim1 = left→right (left low). Swap MCMC cols if needed.
orient_and_scale <- function(points) {
  points <- points %>% left_join(deputy_info, by = "acteur_id")
  left_mask <- points$groupe_code %in% left_groups
  right_mask <- points$groupe_code %in% right_groups
  sep1 <- abs(mean(points$dim1[left_mask], na.rm = TRUE) - mean(points$dim1[right_mask], na.rm = TRUE))
  sep2 <- abs(mean(points$dim2[left_mask], na.rm = TRUE) - mean(points$dim2[right_mask], na.rm = TRUE))
  if (is.finite(sep1) && is.finite(sep2) && sep2 > sep1) {
    tmp <- points$dim1; points$dim1 <- points$dim2; points$dim2 <- tmp
  }
  left_mean <- mean(points$dim1[left_mask], na.rm = TRUE)
  right_mean <- mean(points$dim1[right_mask], na.rm = TRUE)
  if (is.finite(left_mean) && is.finite(right_mean) && left_mean > right_mean) {
    points$dim1 <- -points$dim1
  }
  radius <- max(sqrt(points$dim1^2 + points$dim2^2), na.rm = TRUE)
  points %>% mutate(dim1 = dim1 / radius, dim2 = dim2 / radius,
                    groupe_code = factor(groupe_code, levels = group_order))
}

set.seed(123)
ideal_model <- ideal(
  an_rollcall, d = 2, maxiter = 1000, burnin = 500, thin = 25,
  impute = FALSE, store.item = FALSE, verbose = TRUE
)

points_ideal <- tibble(
  acteur_id = rownames(ideal_model$xbar),
  dim1 = ideal_model$xbar[, 1],
  dim2 = ideal_model$xbar[, 2]
) %>% orient_and_scale()

head(points_ideal)
write_csv(points_ideal, file.path(out_dir, "points_ideal_raw.csv"))



# 4. Pondération APRE et sorties

Pipeline batch recommandé (évite les problèmes de mémoire des start values GLM) :

```bash
python3 scripts/assemblee/prepare_rollcall_l17.py
export AN_MODEL_READY="$PWD/data/assemblee/model_ready"
export AN_OUT_DIR="$PWD/data/assemblee/outputs"
/opt/anaconda3/envs/r_env/bin/Rscript scripts/assemblee/run_ideal_l17.R
python3 scripts/assemblee/postprocess_ideal_l17.py
```

Orientation (comme le Sénat) : `dim1` = gauche→droite (gauche bas). Si MCMC place l'axe G/D sur la colonne 2, swap + flip de signe. Dans l'AN L17, `dim2` résiduelle ≈ majorité/opposition.

Sorties : `data/assemblee/outputs/points_ideal_weighted_full.csv` et les PNG.

